# Breast Cancer Model Comparison
inspect data, tune six leakage-safe pipelines, and compare them with clinically-minded metrics (malignant = positive class)

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import pandas as pd
from tumorclf import data, models, evaluation
from tumorclf.config import Settings
cfg = Settings()

## 1. Data & sanity checks

In [ ]:
frame = data.load_frame()
print(data.quality_report(frame))
frame.iloc[:, [0, 1, 2, -1]].head()

## 2. Split, then tune each pipeline
Scaling lives inside the pipeline, so it is re-fitted on every training fold.

In [ ]:
X_train, X_test, y_train, y_test = data.split(frame, cfg)
fitted = {}
for name, (pipe, space) in models.build_candidates(cfg.seed).items():
    s = models.tune(name, pipe, space, X_train, y_train, cfg)
    fitted[name] = s.best_estimator_
    print(f'{name:22s} CV AUC = {s.best_score_:.4f}')

## 3. Held-out scoreboard

In [ ]:
board = pd.DataFrame({n: evaluation.score_model(m, X_test, y_test) for n, m in fitted.items()}).T
board.sort_values('recall', ascending=False).round(3)

## 4. Which measurements drive the champion?

In [ ]:
champion = board['recall'].astype(float).idxmax()
evaluation.top_features(fitted[champion], X_test, y_test, cfg.seed).plot.barh(title=f'Permutation importance: {champion}');

## 5. Take-aways
* Scaled linear / kernel models match or beat tree ensembles on this small, smooth dataset.
* A single decision tree is the weakest and least stable.
* With a default 0.5 threshold every model still misses some malignant cases; for screening, lower the threshold and accept more false alarms.